In [ ]:
import os
import random
from datasets import load_dataset
import numpy as np
from peft import LoraConfig, TaskType, get_peft_model
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
# Reproducibility Configuration
SEED = 4


def seed_everything(seed=4):
  random.seed(seed)
  os.environ["PYTHONHASHSEED"] = str(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  set_seed(seed)


seed_everything(SEED)

In [ ]:
# Load Model and Tokenizer
model_id = "goldfish-models/eng_latn_1000mb"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)

In [ ]:
# Configure LoRA
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj", "c_fc"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Load and Preprocess XNLI Dataset
xnli_dataset = load_dataset("facebook/xnli", "en")
label_mapping = {0: "entailment", 1: "neutral", 2: "contradiction"}

cls_token = tokenizer.bos_token if tokenizer.bos_token else "<|endoftext|>"
sep_token = tokenizer.eos_token


def formatting_func(examples):
  texts = [
      f"{cls_token}Premise: {p}\nHypothesis: {h}\nLabel:"
      f" {label_mapping[l]}{sep_token}"
      for p, h, l in zip(
          examples["premise"], examples["hypothesis"], examples["label"]
      )
  ]
  tokens = tokenizer(
      texts, truncation=True, max_length=512, padding="max_length"
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens


tokenized_dataset = xnli_dataset.map(
    formatting_func,
    batched=True,
    remove_columns=xnli_dataset["train"].column_names,
)

In [ ]:
# Configure Trainer and Start Fine-Tuning
output_dir = "./goldfish-xnli-lora-final"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False
    ),
)

trainer.train()

In [ ]:
# Save Fine-Tuned Adapter and Tokenizer
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)